<a href="https://www.kaggle.com/code/quddusikashaf/roformer?scriptVersionId=286532796" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Data Processing

In [ ]:
# Based on the paper: "A rotary transformer cross-subject model for continuous estimation of finger joints kinematics and a transfer learning approach for new subjects"
# Dataset: Ninapro DB2 (sEMG, 12 channels, 10 joint angles)
# Architecture: Conv1D -> RoPE -> Multi-Head Attention -> Add&Norm -> Feed Forward -> Add&Norm -> FC (10 angles)
# Requirements: Download Ninapro DB2 from https://ninapro.hevs.ch/ (registration required), process using https://github.com/increase24/Ninapro-dataset-processing
# Assume processed data: sEMG windows (12 x time_steps) and labels (10 joint angles)

# Install Dependencies
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets accelerate wandb tqdm numpy scikit-learn


In [ ]:
import os
import numpy as np
import glob
import scipy.io as scio
from scipy.signal import butter, sosfiltfilt

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random


In [ ]:
def mu_law_normalize(x, mu=255):
    """
    Apply μ-law companding to EMG data.
    Input x should be normalized to [-1,1].
    """
    return np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)

def bandpass_butterworth(emg, lowcut=5, highcut=500, fs=2000, order=4):
    """
    Apply a Butterworth bandpass filter to EMG data.
    
    Args:
    - emg: [time, channels] array
    - lowcut: Low frequency cutoff (Hz)
    - highcut: High frequency cutoff (Hz)
    - fs: Sampling rate (Hz)
    - order: Filter order
    
    Returns:
    - Filtered emg array
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], btype='band', output='sos')
    return sosfiltfilt(sos, emg, axis=0)

def lowpass_butterworth(emg, cutoff=450, fs=2000, order=4):
    """
    Apply a Butterworth low-pass filter to EMG data.
    
    Args:
    - emg: [time, channels] array
    - cutoff: Cutoff frequency (Hz)
    - fs: Sampling rate (Hz)
    - order: Filter order
    
    Returns:
    - Filtered emg array
    """
    nyq = 0.5 * fs
    high = cutoff / nyq
    sos = butter(order, high, btype='low', output='sos')
    return sosfiltfilt(sos, emg, axis=0)

def downsample_emg(emg, fs_original=2000, target_fs=1000):
    """
    Downsample EMG signals using decimation.
    
    Args:
    - emg: [time, channels] array
    - fs_original: Original sampling rate (Hz)
    - target_fs: Target sampling rate (Hz)
    
    Returns:
    - Downsampled emg array
    """
    down_factor = int(fs_original / target_fs)
    return decimate(emg, down_factor, axis=0)

In [ ]:
def process_dataset_for_oneSubject(subject, window_size=400, stride=50, mu=255):
    print(f"Processing subject: {subject}")
    data_paths = sorted(glob.glob(f'/kaggle/input/ninapro-db2/{subject}/{subject}/*.mat'))
    
    emg_all, glove_all, restim_all = [], [], []
    for path in data_paths[:2]:  # use first two sessions
        data = scio.loadmat(path)
        emg_all.append(data['emg'])
        glove_all.append(data['glove'])
        restim_all.append(data['restimulus'])   # repetition labels
    
    emg = np.vstack(emg_all).astype(np.float32)
    glove = np.vstack(glove_all).astype(np.float32)
    restimulus = np.vstack(restim_all).astype(np.int32).squeeze()

    # --- EMG: Apply Butterworth filters ---
    emg = bandpass_butterworth(emg)

    # --- EMG: μ-law normalization ---
    emg = emg / (np.max(np.abs(emg), axis=0, keepdims=True) + 1e-8)
    emg = mu_law_normalize(emg, mu=mu)

    # --- Glove: Min-Max normalization [0,1] ---
    glove_min = np.min(glove, axis=0, keepdims=True)
    glove_max = np.max(glove, axis=0, keepdims=True)
    glove = (glove - glove_min) / (glove_max - glove_min + 1e-8)

    # 🔧 Keep only first 10 glove joints
    cols = [1, 2, 4, 5, 7, 8, 11, 12, 15, 16]
    glove = glove[:, cols]
    glove_min = glove_min[:, cols]
    glove_max = glove_max[:, cols]    

    # --- Sliding window segmentation ---
    emg_windows, glove_windows, restim_windows = [], [], []
    for i in range(0, len(emg) - window_size, stride):
        emg_win = emg[i:i+window_size, :]
        glove_win = glove[i+window_size-1, :]
        restim_win = restimulus[i+window_size-1]  # label at last frame
        emg_windows.append(emg_win)
        glove_windows.append(glove_win)
        restim_windows.append(restim_win)

    emg_windows = np.array(emg_windows)
    glove_windows = np.array(glove_windows)
    restim_windows = np.array(restim_windows)

    # Save processed data
    output_dir = f"/kaggle/working/db2_processed/{subject}"
    os.makedirs(output_dir, exist_ok=True)
    np.savez(f"{output_dir}/data.npz", 
             sEMG=emg_windows, 
             angles=glove_windows, 
             restim=restim_windows)
    print(f"Saved {emg_windows.shape[0]} samples for {subject}")


In [ ]:
base_path = '/kaggle/input/ninapro-db2'
subjects = sorted([folder for folder in os.listdir(base_path)])
for i, subject in enumerate(subjects[:2]):  # first 10 for testing
    process_dataset_for_oneSubject(subject)

# ----------------------------------------------------------
# # Example: first 15 subjects for training, last 5 for validation
# base_path = '/kaggle/input/ninapro-db2'
# subjects = sorted([folder for folder in os.listdir(base_path)])

# for subject in subjects[:10]:   # preprocess first 20 subjects
#     process_dataset_for_oneSubject(subject)
    
# train_subjects = subjects[:7]
# val_subjects   = subjects[7:10]

# train_files = [f"/kaggle/working/db2_processed/{s}/data.npz" for s in train_subjects]
# val_files   = [f"/kaggle/working/db2_processed/{s}/data.npz" for s in val_subjects]



In [ ]:

# class NinaproDataset(Dataset):
#     def __init__(self, npz_files, split='train', split_ratio=0.8):
#         # Load all subjects and concatenate
#         sEMG_all, angles_all = [], []
#         for f in npz_files:
#             data = np.load(f,mmap_mode='r')
#             sEMG_all.append(data['sEMG'])
#             angles_all.append(data['angles'])
#         self.sEMG = np.concatenate(sEMG_all, axis=0)
#         self.angles = np.concatenate(angles_all, axis=0)

#         # Split
#         split_idx = int(split_ratio * len(self.sEMG))
#         if split == 'train':
#             self.sEMG = self.sEMG[:split_idx]
#             self.angles = self.angles[:split_idx]
#         else:
#             self.sEMG = self.sEMG[split_idx:]
#             self.angles = self.angles[split_idx:]

#     def __len__(self):
#         return len(self.sEMG)

#     def __getitem__(self, idx):
#         # Convert [window_size, 12] -> [12, window_size] for CNN/RNN models
#         sEMG = torch.tensor(self.sEMG[idx].T, dtype=torch.float32)
#         angles = torch.tensor(self.angles[idx], dtype=torch.float32)
#         return sEMG, angles


## Previous Dataset Class

In [ ]:
class NinaproDataset(Dataset):
    def __init__(self, npz_files, split='train', split_ratio=0.8):
        sEMG_all, angles_all = [], []

        for f in npz_files:
            data = np.load(f, mmap_mode='r')
            sEMG = data['sEMG']
            angles = data['angles']
            restim = data['restim']

            # Split per restimulus label
            unique_reps = np.unique(restim)
            for rep in unique_reps:
                idx = np.where(restim == rep)[0]
                split_idx = int(split_ratio * len(idx))
                if split == 'train':
                    sEMG_all.append(sEMG[idx[:split_idx]])
                    angles_all.append(angles[idx[:split_idx]])
                else:
                    sEMG_all.append(sEMG[idx[split_idx:]])
                    angles_all.append(angles[idx[split_idx:]])

        # Concatenate across subjects and repetitions
        self.sEMG = np.concatenate(sEMG_all, axis=0)
        self.angles = np.concatenate(angles_all, axis=0)

    def __len__(self):
        return len(self.sEMG)

    def __getitem__(self, idx):
        # Transpose sEMG to [Channels, Time] for Conv1d
        sEMG = torch.tensor(self.sEMG[idx].T, dtype=torch.float32)
        
        current_angle = torch.tensor(self.angles[idx], dtype=torch.float32)
        
        # TRICK: Get the previous angle to calculate velocity
        # If idx is 0 (start of data), just use current_angle (velocity 0)
        prev_idx = max(0, idx - 1)
        prev_angle = torch.tensor(self.angles[prev_idx], dtype=torch.float32)
        
        return sEMG, current_angle, prev_angle

## LSO

In [ ]:
class NinaproDataset(Dataset):
    def __init__(self, npz_files):
        sEMG_all, angles_all = [], []
        for f in npz_files:
            data = np.load(f, mmap_mode='r')
            sEMG_all.append(data['sEMG'])
            angles_all.append(data['angles'])
        self.sEMG = np.concatenate(sEMG_all, axis=0)
        self.angles = np.concatenate(angles_all, axis=0)

    def __len__(self):
        return len(self.sEMG)

    def __getitem__(self, idx):
        sEMG = torch.tensor(self.sEMG[idx].T, dtype=torch.float32)
        angles = torch.tensor(self.angles[idx], dtype=torch.float32)
        return sEMG, angles


In [ ]:

# Config
class Config:
    exp_name = 'roformer_sEMG'
    data_path = '/content/processed_ninapro_db2.npz'  # Processed NPZ file (sEMG and labels)
    save_dir = 'outputs'
    max_seq_len = 400  # Time steps (50ms at 2kHz)
    num_channels = 12  # sEMG channels
    num_angles = 10  # Output joint angles
    batch_size = 32
    lr = 1e-4
    epochs = 50
    seed = 42
    num_heads = 4
    num_layers = 1
    d_model = 128
    d_ff = 256
    dropout = 0.1
    patience = 4

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)


In [ ]:
processed_dir = "/kaggle/working/db2_processed"
npz_files = sorted(glob.glob(f"{processed_dir}/*/data.npz"))

train_dataset = NinaproDataset(npz_files, 'train')
test_dataset = NinaproDataset(npz_files, 'test')

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, num_workers=4 ,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, num_workers=4, shuffle=False)
# train_dataset = NinaproDataset(train_files)
# val_dataset   = NinaproDataset(val_files)

# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
# val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
# test_dataset = val_dataset

# test_loader = val_loader

In [ ]:
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")


## Model Architecture

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from tqdm import tqdm
import random
import os
from transformers import RoFormerModel, RoFormerConfig


In [ ]:

# Step 2: RoPE (Rotary Position Embedding)
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2).float().to(device) / self.dim))
        self.register_buffer("inv_freq", inv_freq)

        # Build here to make `torch.jit.trace` work
        self._set_cos_sin_cache(
            seq_len=max_position_embeddings, device=self.inv_freq.device, dtype=torch.get_default_dtype()
        )

    def _set_cos_sin_cache(self, seq_len, device, dtype):
        self.max_seq_len_cached = seq_len
        t = torch.arange(self.max_seq_len_cached, device=device, dtype=self.inv_freq.dtype)

        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        # Different from paper, but it uses a different permutation in order to obtain the same calculation
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)

    def forward(self, x, seq_len=None):
        # x: [bs, num_attention_heads, seq_len, head_size]
        if seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(seq_len=seq_len, device=x.device, dtype=x.dtype)
        return (
            self.cos_cached[:seq_len].to(dtype=x.dtype),
            self.sin_cached[:seq_len].to(dtype=x.dtype),
        )


In [ ]:

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    # q, k: [batch, seq_len, d_model]
    # cos, sin: [seq_len, d_model]
    cos = cos.unsqueeze(0)  # [1, seq_len, d_model]
    sin = sin.unsqueeze(0)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed



In [ ]:
class RoFormerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # self.activation = nn.GELU()
        self.activation = nn.ReLU()
    def forward(self, src, cos=None, sin=None, src_mask=None, src_key_padding_mask=None):
        q = k = src
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)  # no unsqueeze
        src2 = self.self_attn(q, k, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src


In [ ]:
# Add this import
import torch.quantization as quant

# New: Wrapper for QAT (adds quantization stubs)
class QuantizedRoFormer(nn.Module):
    def __init__(self, model_fp32):
        super().__init__()
        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()
        self.model_fp32 = model_fp32

    def forward(self, x):
        x = self.quant(x)
        x = self.model_fp32(x)
        x = self.dequant(x)
        return x

In [ ]:

class RoFormer(nn.Module):
    def __init__(self, num_channels=12, time_steps=100, num_layers=4, num_heads=8, d_model=128, d_ff=512, num_angles=10, dropout=0.1):
        super().__init__()
        self.conv1d = nn.Conv1d(num_channels, d_model, kernel_size=3, padding=1)  # Conv1D on sEMG channels
        self.rope = RotaryEmbedding(d_model)
        self.layers = nn.ModuleList([RoFormerLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, num_angles)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: [batch, channels, time_steps]
        x = self.conv1d(x)                # [batch, d_model, time_steps]
        x = x.transpose(1, 2)             # [batch, time_steps, d_model]
        x = self.dropout(x)
    
        seq_len = x.shape[1]
        cos, sin = self.rope(x, seq_len=seq_len)
    
        for layer in self.layers:
            x = layer(x, cos, sin)
    
        x = x.mean(dim=1)
        x = self.fc(x)
        return x


# Step 4: Training Loop
model = RoFormer(cfg.num_channels, cfg.max_seq_len, cfg.num_layers, cfg.num_heads, cfg.d_model, cfg.d_ff, cfg.num_angles, cfg.dropout).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

model = model.to(device)


optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.MSELoss()


## Custom Loss

In [ ]:
from tqdm import tqdm

class TemporalVelocityRegularizedLoss(nn.Module):
    def __init__(self, lambda_v=0.1, sequence_dim=1):
        """
        Custom loss for temporal regularization using finite differences.
        
        :param lambda_v: Weighting factor for the velocity loss term.
        :param sequence_dim: The dimension that represents the time sequence 
                             in the output tensor (e.g., 1 if batch_size x seq_len x features).
        """
        super().__init__()
        self.mse = nn.MSELoss()
        self.lambda_v = lambda_v
        self.sequence_dim = sequence_dim

    def forward(self, y_pred, y_target):
        # 1. Position Loss (Standard MSE)
        loss_mse = self.mse(y_pred, y_target)

        # 2. Calculate Predicted Temporal Difference (Velocity)
        # We slice off the last element and the first element along the sequence dimension
        # (y_i+1 - y_i)
        
        # Calculate Delta Y_pred: y_pred[1:] - y_pred[:-1]
        # Example for sequence_dim=1: y_pred[:, 1:, :] - y_pred[:, :-1, :]
        
        y_pred_diff = torch.diff(y_pred, dim=self.sequence_dim)
        
        # 3. Calculate Target Temporal Difference (Velocity)
        # Target must be detached because it is ground truth and should not 
        # contribute to gradients beyond the initial MSE.
        y_target_diff = torch.diff(y_target, dim=self.sequence_dim)
        
        # 4. Velocity Loss (MSE of the Temporal Differences)
        # L_velocity = MSE(Delta Y_pred, Delta Y_target)
        loss_vel_unscaled = self.mse(y_pred_diff, y_target_diff)
        
        # 5. Total Loss
        total_loss = loss_mse + (self.lambda_v * loss_vel_unscaled)
        
        return total_loss


In [ ]:
# Replace model creation section with this
model_fp32 = RoFormer(cfg.num_channels, cfg.max_seq_len, cfg.num_layers, cfg.num_heads, cfg.d_model, cfg.d_ff, cfg.num_angles, cfg.dropout)

# Wrap and prepare for QAT
model = QuantizedRoFormer(model_fp32).to(device)
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')  # Or 'qnnpack' for ARM
# Disable QAT for MultiheadAttention (unsupported)
for layer in model.model_fp32.layers:
    layer.self_attn.qconfig = None
torch.quantization.prepare_qat(model, inplace=True)  # Inserts fake quants

# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     model = torch.nn.DataParallel(model)

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.MSELoss()

## Training

In [ ]:

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_sEMG, batch_angles in tqdm(loader):
        batch_sEMG = batch_sEMG.to(device)
        batch_angles = batch_angles.to(device)
        optimizer.zero_grad()
        outputs = model(batch_sEMG)
        loss = criterion(outputs, batch_angles)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch_sEMG, batch_angles in loader:
            batch_sEMG = batch_sEMG.to(device)
            batch_angles = batch_angles.to(device)
            outputs = model(batch_sEMG)
            loss = criterion(outputs, batch_angles)
            total_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(batch_angles.cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
    r2 = r2_score(all_targets, all_preds)
    return total_loss / len(loader), rmse, r2


In [ ]:
# Replace model_size function with this (for INT8 estimation)
def model_size(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # For quantized model, estimate INT8 size (weights/activations ~1 byte per elem vs. 4 for FP32)
    # Note: This is approximate; actual depends on backend
    is_quantized = any('quantized' in str(type(m)) for m in model.modules())
    bytes_per_param = 1 if is_quantized else 4  
    print(is_quantized)

    total_size_mb = total_params * bytes_per_param / (1024 ** 2)
    trainable_size_mb = trainable_params * bytes_per_param / (1024 ** 2)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total size (MB): {total_size_mb:.2f}")
    print(f"Trainable size (MB): {trainable_size_mb:.2f}")

    return total_params, total_size_mb
model_size(model)

In [ ]:

# Training
best_r = 0.0
patience_counter = 0
for epoch in range(cfg.epochs+30):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, rmse, r2 = validate(model, test_loader, criterion, device)
    print(f'Epoch {epoch+1}/{cfg.epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}')
    if r2 > best_r:
        best_r = r2
        torch.save(model.state_dict(), os.path.join(cfg.save_dir, 'best_roformer.pt'))
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print(f'Early stopping at epoch {epoch+1}')
            break



In [ ]:
# --------------------------------------------------------------
# AFTER TRAINING – Convert to static INT8
# --------------------------------------------------------------
print("Converting QAT model to static INT8...")
model.eval()                     # required for conversion
model = model.cpu()              # move QAT model (with fake quants) to CPU
torch.cuda.empty_cache()

# Convert
quantized_model = torch.quantization.convert(model.cpu(), inplace=False)

# Move quantized model back to GPU (or keep on CPU for inference)
quantized_model = quantized_model.to(device)

# --------------------------------------------------------------
# VALIDATE THE QUANTIZED MODEL
# --------------------------------------------------------------
quant_val_loss, quant_rmse, quant_r2 = validate(
    quantized_model, test_loader, criterion, device
)
print(f"Quantized Val Loss={quant_val_loss:.4f}, "
      f"RMSE={quant_rmse:.4f}, R2={quant_r2:.4f}")

# Size check
model_size(quantized_model)

# Save
torch.save(
    quantized_model.state_dict(),
    os.path.join(cfg.save_dir, "best_roformer_quantized.pt")
)

In [ ]:
# Add this after the training loop
# After training: Convert to fully quantized model
model.eval()  # Switch to eval for conversion
quantized_model = torch.quantization.convert(model, inplace=False)

# Optional: Validate the quantized model
quant_val_loss, quant_rmse, quant_r2 = validate(quantized_model, test_loader, criterion, device)
print(f"Quantized Val Loss={quant_val_loss:.4f}, RMSE={quant_rmse:.4f}, R2={quant_r2:.4f}")

# Check quantized size
model_size(quantized_model)

# Save quantized model
torch.save(quantized_model.state_dict(), os.path.join(cfg.save_dir, 'best_roformer_quantized.pt'))

In [ ]:
# Inference Example
model.load_state_dict(torch.load(os.path.join(cfg.save_dir, 'best_roformer.pt')))
model.eval()


In [ ]:

def predict_angles(sEMG_window):  # sEMG_window: [1, 12, 100]
    sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        angles = model(sEMG_tensor)
    return angles.cpu().numpy()


In [ ]:
def predict_and_compare(sEMG_window, true_angles):  
    """
    sEMG_window: [12, time_steps] or [1, 12, time_steps]
    true_angles: [num_joints] ground-truth glove values
    """
    # Ensure correct shape: [batch, 12, time_steps]
    if sEMG_window.ndim == 2:   # [12, time_steps]
        sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).unsqueeze(0).to(device)
    elif sEMG_window.ndim == 3: # [1, 12, time_steps]
        sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).to(device)
    else:
        raise ValueError("Unexpected sEMG_window shape")

    with torch.no_grad():
        pred_angles = model(sEMG_tensor)

    pred_angles = pred_angles.cpu().numpy().squeeze()
    true_angles = np.array(true_angles)

    # Print comparison
    print("True angles:      ", np.round(true_angles, 3))
    print("Predicted angles: ", np.round(pred_angles, 3))

    return pred_angles


In [ ]:
# Take one batch from DataLoader
for sEMG_batch, angles_batch in train_loader:
    sEMG_sample = sEMG_batch[0].unsqueeze(0).numpy()   # [1, 12, time_steps]
    true_sample = angles_batch[0].numpy()              # [num_joints]
    predict_and_compare(sEMG_sample, true_sample)
    break


In [ ]:
import torch
import torch.nn as nn
import os

# --- IMPORTANT ---
# 1. COPY your model class definitions here:
#
# class Config:
#     ... (your cfg)
#
# class RotaryEmbedding(nn.Module):
#     ... (your definition)
#
# def rotate_half(x):
#     ... (your definition)
#
# def apply_rotary_pos_emb(q, k, cos, sin):
#     ... (your definition)
#
# class RoFormerLayer(nn.Module):
#     ... (your definition)
#
# class RoFormer(nn.Module):
#     ... (your definition)
#
# --- (End of definitions) ---


def export_fp32_model():
    print("Exporting FP32 model to ONNX...")
    
    # --- 1. Load Configuration ---
    # Use the same config as your training
    cfg = Config() 
    device = torch.device('cpu') # Exporting is best done on CPU
    
    # --- 2. Instantiate the FP32 Model ---
    # Use your original RoFormer, NOT the QuantizedRoFormer
    model = RoFormer(
        num_channels=cfg.num_channels, 
        time_steps=cfg.max_seq_len, 
        num_layers=cfg.num_layers, 
        num_heads=cfg.num_heads, 
        d_model=cfg.d_model, 
        d_ff=cfg.d_ff, 
        num_angles=cfg.num_angles, 
        dropout=cfg.dropout
    )
    
    # --- 3. Load Your Trained Weights ---
    # Load the *best_roformer.pt* file you saved during training
    weights_path = os.path.join(cfg.save_dir, 'best_roformer.pt')
    
    # We must load the state_dict from the *inner* model,
    # since you saved it from the QAT wrapper.
    # If you saved the FP32 model, just use: model.load_state_dict(torch.load(weights_path))
    state_dict = torch.load(weights_path, map_location=device)
    
    # Clean keys if they were saved inside the 'model_fp32.' wrapper
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        if k.startswith('model_fp32.'):
            name = k[11:] # remove `model_fp32.` prefix
            new_state_dict[name] = v
        else:
            new_state_dict[k] = v
            
    model.load_state_dict(new_state_dict, strict = False)
    
    # --- 4. Set to Evaluation Mode ---
    # This is CRITICAL. It disables dropout, etc.
    model.eval()
    
    # --- 5. Create a Dummy Input ---
    # The shape must match your model's forward pass: [batch, channels, time_steps]
    # We use batch_size = 1, as this is typical for real-time inference in Unity.
    dummy_input = torch.randn(1, cfg.num_channels, cfg.max_seq_len, device=device)
    
    # --- 6. Export to ONNX ---
    onnx_file_path = "roformer_model_fp32.onnx"
    
    torch.onnx.export(
        model,
        dummy_input,
        onnx_file_path,
        input_names=['input_emg'],    # Name for Unity to find
        output_names=['output_angles'], # Name for Unity to find
        opset_version=12,             # A good stable version for Unity
        export_params=True,
        do_constant_folding=True
    )
    
    print(f"Successfully exported FP32 model to {onnx_file_path}")

if __name__ == '__main__':
    # (Make sure your model definitions are pasted above)
    export_fp32_model()

## Testing Loaded Model

In [ ]:
import torch
import torch.nn as nn
import os
import numpy as np
from collections import OrderedDict


def predict_and_compare(model, device, sEMG_window, true_angles):
    # Ensure correct shape: [batch, 12, time_steps]
    if sEMG_window.ndim == 2:   # [12, time_steps]
        sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).unsqueeze(0).to(device)
    elif sEMG_window.ndim == 3: # [1, 12, time_steps]
        sEMG_tensor = torch.tensor(sEMG_window, dtype=torch.float32).to(device)
    else:
        raise ValueError("Unexpected sEMG_window shape")

    # Run inference
    with torch.no_grad():
        pred_angles = model(sEMG_tensor)

    # Move to CPU and remove batch dimension
    pred_angles = pred_angles.cpu().numpy().squeeze()
    true_angles = np.array(true_angles).squeeze()

    # Print comparison
    print("--- Model Evaluation Check ---")
    print(f"Input Shape: {sEMG_tensor.shape}")
    print(f"Output Shape: {pred_angles.shape}")
    print("------------------------------")
    print("True angles:      ", np.round(true_angles, 3))
    print("Predicted angles: ", np.round(pred_angles, 3))
    print("------------------------------")
    return pred_angles


In [ ]:
def evaluate_loaded_model():
    # --- 1. Load Configuration ---
    try:
        cfg = Config()
    except NameError:
        print("ERROR: 'Config' class not defined. Please paste it at the top.")
        return

    device = torch.device('cpu')  # Use CPU for this check

    # --- 2. Instantiate the FP32 Model ---
    try:
        model = RoFormer(
            num_channels=cfg.num_channels,
            time_steps=cfg.max_seq_len,
            num_layers=cfg.num_layers,
            num_heads=cfg.num_heads,
            d_model=cfg.d_model,
            d_ff=cfg.d_ff,
            num_angles=cfg.num_angles,
            dropout=cfg.dropout
        ).to(device)
    except NameError:
        print("ERROR: 'RoFormer' class not defined. Please paste it at the top.")
        return

    # --- 3. Load Your Trained Weights (with Key Cleaning) ---
    weights_path = os.path.join('/kaggle/input/roformer/pytorch/default/1/best_roformer.pt')
    print(f"Loading weights from: {weights_path}")

    try:
        state_dict = torch.load(weights_path, map_location=device)
    except FileNotFoundError:
        print(f"ERROR: Weights file not found at {weights_path}")
        print("Please check your 'cfg.save_dir' path.")
        return

    # Clean keys if they were saved inside the 'model_fp32.' wrapper
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        if k.startswith('model_fp32.'):
            name = k[11:]  # remove `model_fp32.` prefix
            new_state_dict[name] = v
        else:
            new_state_dict[k] = v

    model.load_state_dict(new_state_dict, strict=False)
    print("Successfully loaded and cleaned weights.")

    # --- 4. Set to Evaluation Mode ---
    model.eval()

    # --- 5. Load a Real Data Sample ---
    try:
        # NOTE: You must provide your Dataset/DataLoader logic
        # This is an example, adjust it to match your data pipeline
        # Ensure 'train_dataset' is an instance of 'YourEMGDataset'
        # train_dataset = YourEMGDataset(data_path=..., labels_path=...)
        # train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

        # Get one batch from your DataLoader
        sEMG_batch, angles_batch = next(iter(train_loader))

        # Take the first sample from the batch
        sEMG_sample = sEMG_batch[0].numpy()   # Shape: [12, time_steps]
        true_sample = angles_batch[0].numpy()  # Shape: [num_joints]

    except NameError:
        print("\nERROR: 'train_loader' or 'YourEMGDataset' is not defined.")
        print("Please paste your Dataset/DataLoader code at the top to run the comparison.")
        print("\n--- Evaluation check skipped. ---")
        return
    except Exception as e:
        print(f"\nERROR: Failed to load data from DataLoader: {e}")
        print("Please check your data loading pipeline.")
        print("\n--- Evaluation check skipped. ---")
        return

    # --- 6. Predict and Compare ---
    predict_and_compare(model, device, sEMG_sample, true_sample)


if __name__ == '__main__':
    evaluate_loaded_model()
